In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import RadioButtons, IntSlider, FloatSlider, VBox, HBox, Layout, HTML
from IPython.display import display

plt.ioff()

# ==============================================================================
# USAGE
#
# GENERAL TRANSITIONAL ALL-POLE FILTER
#
# Two different reference all-pole filters are selected:
#
#       Reference Filter 1
#       Reference Filter 2
#
# Available filters:
#
#       Butterworth
#       Chebyshev I
#       Bessel-Thomson
#
# Transitional poles:
#
#       pk(m) = [pk^(2)]^m [pk^(1)]^(1-m)
#
#       0 <= m <= 1
#
# Therefore:
#
#       m = 0  -> Reference Filter 1
#       m = 1  -> Reference Filter 2
#
# Intermediate values of m generate intermediate pole locations and
# intermediate frequency-domain behavior.
#
# Interactive controls:
#
#       N : filter order, 1,...,10
#       m : transition parameter, 0,...,1
#
# Displayed quantities:
#
#       Pole Geometry
#       Magnitude Response
#       Phase Response
#       Group Delay
#
# In every response plot, three curves are displayed:
#
#       Reference Filter 1
#       Transitional Filter
#       Reference Filter 2
#
# ==============================================================================

# ==============================================================================
# IMPORTANT NOTE ON THE AVAILABLE REFERENCE FILTERS
# ==============================================================================
#
# Only Butterworth, Chebyshev I and Bessel-Thomson filters are included.
#
# The transitional-filter relation
#
#       pk = [pk^(2)]^m [pk^(1)]^(1-m)
#
# defines the transition exclusively through the FILTER POLES.
#
# Butterworth, Chebyshev I and Bessel-Thomson are all-pole prototypes, so the
# interpolation can be applied naturally to them.
#
# Chebyshev II and elliptic filters are intentionally excluded because their
# transfer functions also contain finite transmission zeros. Interpolating
# only their poles would not, in general, reproduce the complete reference
# filters at m = 0 or m = 1. An additional interpolation rule for the
# transmission zeros would therefore be required. Such a rule lies outside
# the transitional-filter formulation considered here.
# ==============================================================================

# ==============================================================================
# JUPYTER DISPLAY SETTINGS
# ==============================================================================

display(HTML("""
<style>

.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.output,
.output_area,
.output_subarea,
.output_scroll {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.jupyter-widgets,
.widget-box,
.widget-html,
.widget-html-content {
    overflow: visible !important;
    max-height: none !important;
}

.jp-Cell-outputWrapper {
    overflow: visible !important;
}

</style>
"""))

# ==============================================================================
# FIXED PARAMETERS
# ==============================================================================

Ap = 1.0

omega = np.logspace(-2, 2, 5000)

filter_options = ['Butterworth', 'Chebyshev I', 'Bessel-Thomson']

# ==============================================================================
# REFERENCE FILTER POLES
# ==============================================================================

def get_reference_poles(filter_name, N):

    if filter_name == 'Butterworth':
        z, poles, k = signal.buttap(N)

    elif filter_name == 'Chebyshev I':
        z, poles, k = signal.cheb1ap(N, Ap)

    elif filter_name == 'Bessel-Thomson':
        z, poles, k = signal.besselap(N, norm='phase')

    return np.asarray(poles, dtype=complex)

# ==============================================================================
# ORGANIZE POLES
# ==============================================================================

def organize_poles(poles):

    tolerance = 1e-10

    upper = [p for p in poles if p.imag > tolerance]

    real = [p for p in poles if abs(p.imag) <= tolerance]

    upper = sorted(upper, key=lambda z: np.angle(z))

    real = sorted(real, key=lambda z: z.real)

    return upper, real

# ==============================================================================
# POLE INTERPOLATION
#
# If:
#
#       p1 = r1 exp(jθ1)
#       p2 = r2 exp(jθ2)
#
# then:
#
#       p(m) = r1^(1-m) r2^m exp[j((1-m)θ1 + mθ2)]
#
# ==============================================================================

def interpolate_pole(p1, p2, m):

    r1 = np.abs(p1)

    r2 = np.abs(p2)

    theta1 = np.angle(p1)

    theta2 = np.angle(p2)

    radius = np.exp((1.0 - m) * np.log(r1) + m * np.log(r2))

    theta = (1.0 - m) * theta1 + m * theta2

    return radius * np.exp(1j * theta)

# ==============================================================================
# TRANSITIONAL POLES
# ==============================================================================

def construct_transitional_poles(filter1, filter2, N, m):

    poles1 = get_reference_poles(filter1, N)

    poles2 = get_reference_poles(filter2, N)

    upper1, real1 = organize_poles(poles1)

    upper2, real2 = organize_poles(poles2)

    transitional_upper = []

    for p1, p2 in zip(upper1, upper2):
        transitional_upper.append(interpolate_pole(p1, p2, m))

    transitional_real = []

    for p1, p2 in zip(real1, real2):
        transitional_real.append(interpolate_pole(p1, p2, m))

    transitional_poles = []

    transitional_poles.extend(transitional_upper)

    transitional_poles.extend(transitional_real)

    transitional_poles.extend([np.conjugate(p) for p in transitional_upper])

    transitional_poles = np.asarray(transitional_poles, dtype=complex)

    return poles1, poles2, transitional_poles

# ==============================================================================
# TRANSFER FUNCTION FROM POLES
#
# The numerator is selected so that:
#
#       H(0) = 1
#
# ==============================================================================

def transfer_function_from_poles(poles):

    denominator = np.poly(poles)

    denominator = np.real_if_close(denominator, tol=1000).real

    numerator = np.array([denominator[-1]])

    return numerator, denominator

# ==============================================================================
# REFERENCE FILTER TRANSFER FUNCTION
# ==============================================================================

def construct_reference_filter(filter_name, N):

    poles = get_reference_poles(filter_name, N)

    b, a = transfer_function_from_poles(poles)

    return b, a, poles

# ==============================================================================
# TRANSITIONAL TRANSFER FUNCTION
# ==============================================================================

def construct_transitional_filter(filter1, filter2, N, m):

    poles1, poles2, transitional_poles = construct_transitional_poles(filter1, filter2, N, m)

    b, a = transfer_function_from_poles(transitional_poles)

    return b, a, poles1, poles2, transitional_poles

# ==============================================================================
# RESPONSE CALCULATION
# ==============================================================================

def calculate_filter_response(b, a):

    _, H = signal.freqs(b, a, worN=omega)

    magnitude = np.abs(H)

    phase = np.unwrap(np.angle(H))

    phase_deg = np.rad2deg(phase)

    group_delay = -np.gradient(phase, omega)

    return magnitude, phase_deg, group_delay

# ==============================================================================
# COMPLETE RESPONSE CALCULATION
# ==============================================================================

def calculate_all(filter1, filter2, N, m):

    b1, a1, poles1 = construct_reference_filter(filter1, N)

    b2, a2, poles2 = construct_reference_filter(filter2, N)

    bt, at, poles1_check, poles2_check, transitional_poles = construct_transitional_filter(filter1, filter2, N, m)

    mag1, phase1, gd1 = calculate_filter_response(b1, a1)

    mag2, phase2, gd2 = calculate_filter_response(b2, a2)

    magt, phaset, gdt = calculate_filter_response(bt, at)

    return {
        'poles1': poles1,
        'poles2': poles2,
        'poles_transitional': transitional_poles,
        'mag1': mag1,
        'mag2': mag2,
        'magt': magt,
        'phase1': phase1,
        'phase2': phase2,
        'phaset': phaset,
        'gd1': gd1,
        'gd2': gd2,
        'gdt': gdt
    }

# ==============================================================================
# DESCRIPTION
# ==============================================================================

description = HTML("""
<div style="
    border:1px solid #9ec9f5;
    border-radius:7px;
    padding:9px 11px;
    margin:0px 0px 8px 0px;
    font-size:12px;
    line-height:1.50;
    background-color:#f7fbff;
    width:1450px;
    max-width:1450px;
    box-sizing:border-box;
">

<b>General Transitional All-Pole Filter</b><br>

Select two different all-pole reference approximations and vary the
transition parameter <b>m</b> between 0 and 1.

At <b>m = 0</b> the transitional filter coincides with Reference Filter 1,
whereas at <b>m = 1</b> it coincides with Reference Filter 2.

<br>

<b>Interpretation:</b>
The pole locations and the corresponding magnitude, phase and group-delay
responses evolve continuously between the two selected reference
approximations. The two reference responses are retained in the plots so that
the movement of the transitional response between the two limiting cases can
be observed directly.

Chebyshev II and elliptic filters are not included because they contain
finite transmission zeros, whereas the transitional relation considered here
interpolates only pole locations.

</div>
""", layout=Layout(width='1460px', max_width='1460px'))

# ==============================================================================
# REFERENCE FILTER SELECTORS
# ==============================================================================

reference1_selector = RadioButtons(options=filter_options, value='Butterworth', description='', layout=Layout(width='155px'))

reference2_selector = RadioButtons(options=filter_options, value='Bessel-Thomson', description='', layout=Layout(width='155px'))

# ==============================================================================
# DISPLAY SELECTOR
# ==============================================================================

display_selector = RadioButtons(options=['Pole Geometry', 'Magnitude Response', 'Phase Response', 'Group Delay'], value='Pole Geometry', description='', layout=Layout(width='155px'))

# ==============================================================================
# PANEL TITLES
# ==============================================================================

reference1_title = HTML('<div style="font-size:13px;font-weight:bold;margin-bottom:5px;">Reference Filter 1</div>')

reference2_title = HTML('<div style="font-size:13px;font-weight:bold;margin-bottom:5px;">Reference Filter 2</div>')

display_title = HTML('<div style="font-size:13px;font-weight:bold;margin-bottom:5px;">Displayed Quantity</div>')

# ==============================================================================
# CONTROL PANELS
# ==============================================================================

control_width = '185px'

reference1_panel = VBox([reference1_title, reference1_selector], layout=Layout(width=control_width, min_width=control_width, max_width=control_width, border='1px solid #cccccc', padding='8px', align_items='flex-start'))

reference2_panel = VBox([reference2_title, reference2_selector], layout=Layout(width=control_width, min_width=control_width, max_width=control_width, border='1px solid #cccccc', padding='8px', align_items='flex-start'))

display_panel = VBox([display_title, display_selector], layout=Layout(width=control_width, min_width=control_width, max_width=control_width, border='1px solid #cccccc', padding='8px', align_items='flex-start'))

radio_column = VBox([reference1_panel, reference2_panel, display_panel], layout=Layout(width='195px', min_width='195px', max_width='195px', align_items='flex-start'))

# ==============================================================================
# SLIDERS
# ==============================================================================

order_slider = IntSlider(value=4, min=1, max=10, step=1, description='Order N:', continuous_update=True, style={'description_width':'65px'}, layout=Layout(width='330px'))

m_slider = FloatSlider(value=0.50, min=0.0, max=1.0, step=0.01, description='Transition m:', continuous_update=True, readout=True, readout_format='.2f', style={'description_width':'90px'}, layout=Layout(width='430px'))

# ==============================================================================
# INFORMATION PANEL
# ==============================================================================

info_html = HTML(layout=Layout(width='285px', max_width='285px'))

# ==============================================================================
# CREATE FIGURE ONCE
# ==============================================================================

fig, ax = plt.subplots(figsize=(8.3, 5.4))

reference1_line, = ax.plot([], [], linewidth=2.0, color='blue')

reference2_line, = ax.plot([], [], linewidth=2.0, color='green')

transitional_line, = ax.plot([], [], linewidth=2.4, color='red')

reference1_pole_line, = ax.plot([], [], 'bo', markersize=7, markerfacecolor='none', markeredgewidth=1.6)

reference2_pole_line, = ax.plot([], [], 'go', markersize=7, markerfacecolor='none', markeredgewidth=1.6)

transitional_pole_line, = ax.plot([], [], 'ro', markersize=7)

horizontal_axis = ax.axhline(0.0, color='black', linewidth=0.8)

vertical_axis = ax.axvline(0.0, color='black', linewidth=0.8)

ax.grid(True, linestyle=':', alpha=0.5)

ax.tick_params(axis='both', labelsize=9)

fig.subplots_adjust(left=0.13, right=0.76, bottom=0.15, top=0.88)

fig.canvas.header_visible = False

fig.canvas.toolbar_visible = False

fig.canvas.resizable = False

fig.canvas.layout.width = '900px'

fig.canvas.layout.height = '530px'

# ==============================================================================
# SELECTION GUARD
# ==============================================================================

selection_guard = False

# ==============================================================================
# ENFORCE DIFFERENT REFERENCE FILTERS
# ==============================================================================

def enforce_different_references(source):

    global selection_guard

    if selection_guard:
        return

    selection_guard = True

    if reference1_selector.value == reference2_selector.value:

        alternatives = [name for name in filter_options if name != reference1_selector.value]

        if source == 'reference1':
            reference2_selector.value = alternatives[0]

        else:
            reference1_selector.value = alternatives[0]

    selection_guard = False

# ==============================================================================
# INFORMATION PANEL UPDATE
# ==============================================================================

def update_info(filter1, filter2, N, m, transitional_poles):

    sorted_poles = sorted(transitional_poles, key=lambda z: (-z.imag, z.real))

    pole_text = '<br>'.join([f'p{k + 1} = {p.real:+.5f} {p.imag:+.5f}j' for k, p in enumerate(sorted_poles)])

    if m <= 1e-12:
        state_text = f'Identical to {filter1}'

    elif m >= 1.0 - 1e-12:
        state_text = f'Identical to {filter2}'

    else:
        state_text = 'Intermediate transitional filter'

    info_html.value = f"""
    <div style="
        border:1px solid #cccccc;
        border-radius:7px;
        padding:9px 10px;
        font-size:12px;
        line-height:1.58;
        background:white;
        width:280px;
        box-sizing:border-box;
    ">

    <b>Reference Filter 1:</b><br>
    <span style="color:#0066cc;">{filter1}</span><br><br>

    <b>Reference Filter 2:</b><br>
    <span style="color:#0066cc;">{filter2}</span>

    <div style="margin-top:7px;padding-top:6px;border-top:1px solid #eeeeee;">

    <b>Filter order:</b>
    <span style="color:#0066cc;">N = {N}</span><br>

    <b>Transition parameter:</b>
    <span style="color:#0066cc;">m = {m:.2f}</span>

    </div>

    <div style="margin-top:7px;padding-top:6px;border-top:1px solid #eeeeee;">

    <b>Current state:</b><br>
    <span style="color:#0066cc;">{state_text}</span>

    </div>

    <div style="margin-top:7px;padding-top:6px;border-top:1px solid #eeeeee;">

    <b>Transitional poles:</b><br>
    <span style="color:#0066cc;">
    {pole_text}
    </span>

    </div>

    <div style="margin-top:7px;padding-top:6px;border-top:1px solid #eeeeee;">

    <b>Interpretation:</b><br>
    m = 0 gives Reference Filter 1.<br>
    m = 1 gives Reference Filter 2.<br>
    Intermediate values continuously interpolate the pole radii, pole angles
    and the corresponding filter response.

    </div>

    </div>
    """

# ==============================================================================
# EXTERNAL LEGEND
# ==============================================================================

def create_external_legend(filter1, filter2, m, pole_mode=False):

    if pole_mode:

        handles = [reference1_pole_line, transitional_pole_line, reference2_pole_line]

    else:

        handles = [reference1_line, transitional_line, reference2_line]

    labels = [
        f'Reference Filter 1\n({filter1})',
        f'Transitional Filter\n(m = {m:.2f})',
        f'Reference Filter 2\n({filter2})'
    ]

    legend = ax.legend(handles, labels, loc='center left', bbox_to_anchor=(1.04, 0.5), borderaxespad=0.0, fontsize=8, frameon=True, title='Legend', title_fontsize=9, labelspacing=1.4, handlelength=2.8)

    return legend

# ==============================================================================
# MAIN UPDATE FUNCTION
# ==============================================================================

def update_plot(change=None):

    filter1 = reference1_selector.value

    filter2 = reference2_selector.value

    N = order_slider.value

    m = m_slider.value

    data = calculate_all(filter1, filter2, N, m)

    selected = display_selector.value

    poles1 = data['poles1']

    poles2 = data['poles2']

    transitional_poles = data['poles_transitional']

    # --------------------------------------------------------------------------
    # RESET VISIBILITY
    # --------------------------------------------------------------------------

    reference1_line.set_visible(False)

    reference2_line.set_visible(False)

    transitional_line.set_visible(False)

    reference1_pole_line.set_visible(False)

    reference2_pole_line.set_visible(False)

    transitional_pole_line.set_visible(False)

    horizontal_axis.set_visible(False)

    vertical_axis.set_visible(False)

    old_legend = ax.get_legend()

    if old_legend is not None:
        old_legend.remove()

    # --------------------------------------------------------------------------
    # POLE GEOMETRY
    # --------------------------------------------------------------------------

    if selected == 'Pole Geometry':

        reference1_pole_line.set_visible(True)

        reference2_pole_line.set_visible(True)

        transitional_pole_line.set_visible(True)

        horizontal_axis.set_visible(True)

        vertical_axis.set_visible(True)

        reference1_pole_line.set_data(np.real(poles1), np.imag(poles1))

        reference2_pole_line.set_data(np.real(poles2), np.imag(poles2))

        transitional_pole_line.set_data(np.real(transitional_poles), np.imag(transitional_poles))

        all_poles = np.concatenate([poles1, poles2, transitional_poles])

        max_real = np.max(np.abs(np.real(all_poles)))

        max_imag = np.max(np.abs(np.imag(all_poles)))

        limit = max(1.2, 1.20 * max(max_real, max_imag))

        ax.set_xscale('linear')

        ax.set_xlim(-limit, 0.25 * limit)

        ax.set_ylim(-limit, limit)

        ax.set_aspect('equal', adjustable='box')

        ax.set_xlabel('Re{s}', fontsize=10)

        ax.set_ylabel('Im{s}', fontsize=10)

        ax.set_title(f'{filter1} → {filter2}: Transitional Pole Geometry', fontsize=13, fontweight='bold', pad=8)

        create_external_legend(filter1, filter2, m, pole_mode=True)

    # --------------------------------------------------------------------------
    # MAGNITUDE RESPONSE
    #
    # The vertical range is intentionally fixed at [0, 1.25].
    # This gives enough room for passband peaks while preserving a constant
    # scale as N and m are varied.
    # --------------------------------------------------------------------------

    elif selected == 'Magnitude Response':

        reference1_line.set_visible(True)

        reference2_line.set_visible(True)

        transitional_line.set_visible(True)

        reference1_line.set_data(omega, data['mag1'])

        reference2_line.set_data(omega, data['mag2'])

        transitional_line.set_data(omega, data['magt'])

        ax.set_aspect('auto')

        ax.set_xscale('log')

        ax.set_xlim(0.01, 100.0)

        ax.set_ylim(0.0, 1.25)

        ax.set_xlabel('Angular Frequency ω (rad/s)', fontsize=10)

        ax.set_ylabel('|H(jω)|', fontsize=10)

        ax.set_title(f'{filter1} → {filter2}: Transitional Magnitude Response', fontsize=13, fontweight='bold', pad=8)

        create_external_legend(filter1, filter2, m)

    # --------------------------------------------------------------------------
    # PHASE RESPONSE
    # --------------------------------------------------------------------------

    elif selected == 'Phase Response':

        reference1_line.set_visible(True)

        reference2_line.set_visible(True)

        transitional_line.set_visible(True)

        horizontal_axis.set_visible(True)

        reference1_line.set_data(omega, data['phase1'])

        reference2_line.set_data(omega, data['phase2'])

        transitional_line.set_data(omega, data['phaset'])

        ax.set_aspect('auto')

        ax.set_xscale('log')

        ax.set_xlim(0.01, 100.0)

        phase_min = min(np.min(data['phase1']), np.min(data['phase2']), np.min(data['phaset']))

        ax.set_ylim(1.05 * phase_min, 10.0)

        ax.set_xlabel('Angular Frequency ω (rad/s)', fontsize=10)

        ax.set_ylabel('Phase (degrees)', fontsize=10)

        ax.set_title(f'{filter1} → {filter2}: Transitional Phase Response', fontsize=13, fontweight='bold', pad=8)

        create_external_legend(filter1, filter2, m)

    # --------------------------------------------------------------------------
    # GROUP DELAY
    # --------------------------------------------------------------------------

    elif selected == 'Group Delay':

        reference1_line.set_visible(True)

        reference2_line.set_visible(True)

        transitional_line.set_visible(True)

        horizontal_axis.set_visible(True)

        gd1 = np.array(data['gd1'], copy=True)

        gd2 = np.array(data['gd2'], copy=True)

        gdt = np.array(data['gdt'], copy=True)

        gd1[np.abs(gd1) > 100.0] = np.nan

        gd2[np.abs(gd2) > 100.0] = np.nan

        gdt[np.abs(gdt) > 100.0] = np.nan

        reference1_line.set_data(omega, gd1)

        reference2_line.set_data(omega, gd2)

        transitional_line.set_data(omega, gdt)

        ax.set_aspect('auto')

        ax.set_xscale('log')

        ax.set_xlim(0.01, 100.0)

        finite_values = np.concatenate([gd1[np.isfinite(gd1)], gd2[np.isfinite(gd2)], gdt[np.isfinite(gdt)]])

        if len(finite_values) > 0:
            gd_max = max(1.0, 1.15 * np.max(finite_values))
        else:
            gd_max = 1.0

        ax.set_ylim(0.0, gd_max)

        ax.set_xlabel('Angular Frequency ω (rad/s)', fontsize=10)

        ax.set_ylabel('Group Delay τ(ω)', fontsize=10)

        ax.set_title(f'{filter1} → {filter2}: Transitional Group Delay', fontsize=13, fontweight='bold', pad=8)

        create_external_legend(filter1, filter2, m)

    # --------------------------------------------------------------------------
    # GRID
    # --------------------------------------------------------------------------

    ax.grid(True, which='both', linestyle=':', alpha=0.5)

    # --------------------------------------------------------------------------
    # INFORMATION PANEL
    # --------------------------------------------------------------------------

    update_info(filter1, filter2, N, m, transitional_poles)

    # --------------------------------------------------------------------------
    # REDRAW
    # --------------------------------------------------------------------------

    fig.canvas.draw_idle()

# ==============================================================================
# REFERENCE FILTER CALLBACKS
# ==============================================================================

def update_reference1(change=None):

    enforce_different_references('reference1')

    update_plot()

def update_reference2(change=None):

    enforce_different_references('reference2')

    update_plot()

# ==============================================================================
# CALLBACKS
# ==============================================================================

reference1_selector.observe(update_reference1, names='value')

reference2_selector.observe(update_reference2, names='value')

order_slider.observe(update_plot, names='value')

m_slider.observe(update_plot, names='value')

display_selector.observe(update_plot, names='value')

# ==============================================================================
# INITIALIZE
# ==============================================================================

update_plot()

# ==============================================================================
# INFORMATION COLUMN
# ==============================================================================

information_column = VBox([info_html], layout=Layout(width='295px', min_width='295px', max_width='295px', align_items='flex-start'))

# ==============================================================================
# SLIDER PANEL
# ==============================================================================

slider_panel = VBox([order_slider, m_slider], layout=Layout(width='900px', align_items='center', margin='-5px 0px 0px 0px'))

# ==============================================================================
# PLOT COLUMN
# ==============================================================================

plot_column = VBox([fig.canvas, slider_panel], layout=Layout(width='900px', min_width='900px', max_width='900px', align_items='center'))

# ==============================================================================
# MAIN LAYOUT
# ==============================================================================

main_layout = HBox([radio_column, information_column, plot_column], layout=Layout(width='1400px', align_items='flex-start', justify_content='flex-start'))

# ==============================================================================
# DISPLAY
# ==============================================================================

display(description)

display(main_layout)